<span style="font-size: 30px;">Prepoznavanje tipa ličnosti na osnovu rukopisa</span>

<span style="font-size: 20px;">Uvod</span>

Grafologija je metod identifikacije, procene i razumevanja osobina ljudi kroz poteze i obrasce koje otkriva rukopis. Prema naučnicima i psiholozima rukopis otkriva pravu ličnost, uključujući strahove, iskrenost, mehanizme odbrane i mnoge druge. Iako je ljudska intervencija u analizi rukopisa bila efikasna skupa je i sklona greškama. Stoga se pokušavamo da razvijemo sistem koji može predvideti osobine ličnosti uz pomoć mašinskog učenja. 

<span style="font-size: 20px;">Motivacija</span>

Analiza rukopisa je jedna od mnogih metoda za razumevanje psihologije osobe. Primena je veoma široka, te se ovaj metod može koristiti u svrhe boljeg razumevanja ličnosti prilikom psihoterapije, analize zakonskih prestupnika, pravljenje boljeg profila ličnosti prilikom zapošljavanja i slično. Grafološki izveštaj treba da se koristi u kombinaciji sa ostalim alatima kao što su provera prošlosti ličnosti, rađenje testova inteligencije i testova prepoznavanja određenih osobina,... Analiza rukopisa je svakako brža, preciznija i bolje identifikuje obrasce od vizuelne inspekcije. Štaviše, analiza uz pomoć mašinskog učenja je efikasnija i lišena ljudskih grešaka.

<span style="font-size: 20px;">Opis problema</span>


Kako je cilj istraživanja da na osnovu rukopisa prepoznamo tip ličnosti, analiziraćemo slike rukopisa nasumično odabranih ljudi deleći ih u 5 grupa prepoznavajući određene osobine u načinu rukopisa. Osobine su:

<span style="font-size: 18px;">1. Prijatnost</span>

<span style="font-size: 18px;">2. Savesnost</span>

<span style="font-size: 18px;">3. Otvorenost</span>

<span style="font-size: 18px;">4. Neuroticizam</span>

<span style="font-size: 18px;">5. Ekstraverzija</span>

<img src="https://miro.medium.com/v2/resize:fit:720/0*Udr3RKnpXoQdf-ud" alt="Opis parametara i osobina">

Parametri koji će biti analizirani u istraživanju su:

 
<span style="font-size: 18px;">1. Gornja margina</span>

<span style="font-size: 18px;">2. Pritisak olovke</span>

<span style="font-size: 18px;">3. Ugao osnovne linije</span>

<span style="font-size: 18px;">4. Veličina slova</span>

<span style="font-size: 18px;">5. Razmak između redova</span>

<span style="font-size: 18px;">6. Razmak između slova</span>

<span style="font-size: 18px;">7. Nagib slova</span>

<img src="https://miro.medium.com/v2/resize:fit:786/0*2MXhN2gKmRXinbJA" alt="Opis parametara i osobina">

<span style="font-size: 20px;">Metodologija korišćenja</span>

Metode koje će biti korištene su KNN algoritam i SVM algoritam, kako su se ta dva algoritma pokazala kao najtačnija algoritma u sličnim istraživanja. 

<span style="font-size: 20px;">Priprema slika</span>

Koristimo dataset od 207 slika podeljenih u 5 grupa, na osnovu gore navedenih osobina. Cilj predobrade je da podaci o slici budu pogodni za ekstrakciju karakteristika za koje smo usvojili metode u nastavku. Slike su isečene i sačuvane kao PNG slike. Širina svih slika je 850px, dok je visina u skladu sa sadržajem teksta na slici. Koristimo PNG format umesto JPG zbog minimalnih gubitaka.

In [ ]:
import cv2
import os
import numpy as np
import svm
import knn
import extract
import matplotlib.pyplot as plt


# Putanja do slika za treinranje
folder_path = "C:/Users/Antonela/Desktop/Fakuktet/3/MITNOP/Handwriting11/dataset/training_set"

# Putanja do slika bez suma i sa binarizacijom
filtered_folder_path = "C:/Users/Antonela/Desktop/Fakuktet/3/MITNOP/Handwriting11/dataset/filtered_training_set"

#Putanja do slika nakon dilatacije, konture i afine transformacije
dialation_folder_path = "C:/Users/Antonela/Desktop/Fakuktet/3/MITNOP/Handwriting11/dataset/final_training_set"

# Gde cuvamo slike i oznake osobina
images = []
labels = []

# Gde cuvamo slike posle uklanjanja suma
filtered_images = []

#Finalno obradjene slike
final_images = []

Šum sa slike se definiše kao nasumična varijacija osvetljenosti i boje na slici. Kako bismo minimizovali neželjeni šum koristimo Gaussianov filter. Takođee konverzija u nijanse sive i binarizacija su važni delovi procesa. Ovu akciju izvršavamo pomoću invertovanog globalnog praga.
Konverzija se vrši pretvaranjem crvene, plave i žute u određene nijanse sive, a zatim se takva slika konvertuje u binarni niz gde je 0 bela boja,a 255 crna.

In [ ]:
for folder in os.listdir(folder_path):
    folder_full_path = os.path.join(folder_path, folder)
    filtered_folder_full_path = os.path.join(filtered_folder_path, folder)
    dialation_folder_full_path = os.path.join(dialation_folder_path, folder)


    # Prolazak kroz sve slike
    for image_name in os.listdir(folder_full_path):
        image_path = os.path.join(folder_full_path, image_name)

        
        # Ucitavanje slike
        image = cv2.imread(image_path)
        
        # Eliminisanje šuma
        filtered_image = cv2.GaussianBlur(image, (5, 5), 0)
        
        # Konverzija u sivu skaliranu verziju
        gray_image = cv2.cvtColor(filtered_image, cv2.COLOR_BGR2GRAY)
        
        # Binarizacija koristeci globalni inverzni prag
        _, binary_image = cv2.threshold(gray_image, 120, 255, cv2.THRESH_BINARY_INV)

<img src="https://miro.medium.com/v2/resize:fit:640/0*BTfFeBhryxXdZvNF" alt="Opis parametara i osobina">

Nakon ovog koraka slika se ispravlja korišćenjem dilatacije, konture i affine transformacije koristeći OpenCV biblioteku. Ovo će dati bolje rezultate sa daljim operacijama korišćenjem horizontalne projekcije slike za izdvajanje linija rukopisa.

In [ ]:
# Dilatacija
kernel = np.ones((5, 100), np.uint8)
dilated_image = cv2.dilate(binary_image, kernel, iterations=1)

# Pronalaženje kontura
contours, _ = cv2.findContours(dilated_image, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# Sortiranje kontura po površini
contours = sorted(contours, key=cv2.contourArea, reverse=True)

# Pronalaženje najveće konture
largest_contour = contours[0]

# Izdvajanje pravougaonika koji obuhvata najveću konturu
x, y, w, h = cv2.boundingRect(largest_contour)
        

# Ispravljanje slike koristeći afine transformacije
points1 = np.float32([[x, y], [x + w, y], [x, y + h]])
points2 = np.float32([[0, 0], [w, 0], [0, h]])
transform_matrix = cv2.getAffineTransform(points1, points2)
corrected_image = cv2.warpAffine(binary_image, transform_matrix, (w, h))

<img src="https://miro.medium.com/v2/resize:fit:640/0*er1aXbeNSy0tEt79" alt="Opis parametara i osobina">

<span style="font-size: 20px;">Parametri karakteristika rukopisa</span>

Kako je već navedeno parametre koje ćemo posmatrati su gornja margina, pritisak olovke, ugao osnovne linije, veličina slova, razmak izmedju redova, razmak izmedju slova i nagib slova. Kreirajući poseban fajl extract.py implementiramo funkcije za ekstraktaciju parametara 

<span style="font-size: 17px;">1. Ugao osnovne linije</span>

Ugao osnovne linije pokazuje da li se tekst kreće ka gore, ka centru ili ka dole. Na osnovu ovoga zaključujemo koliko dobro osoba podnosi mešavinu uticaja intelektualnih, društvenih i instinktivnih nagona.

<span style="font-size: 17px;">2. Veličina slova</span>

Da bi se odredila veličina slova uzorka posmatramo slova na sredini rečenice. Ona bi trebalo da su veličine oko 3,17mm. Sve iznad toga se smatra većim od normalnog, a ispod manjim od normalnog. 

<span style="font-size: 17px;">3. Razmak između redova</span>

Količina prostora koju osoba ostavlja između redova daje naznake o uređenosti i jasnoći njegovog razmišljanja, kao i količine interakcije koju želi da ima sa svojom okolinom.

<span style="font-size: 17px;">4. Razmak između reči</span>

Prostor između napisanih reči predstavlja distancu koju bi osoba želela da održi između sebe i društva u celini. Između reči leži udaljenost koja mu je potrebna za emocionalnu udobnost sa drugima.

<span style="font-size: 17px;">5. Gornja margina</span>

Gornja margina prikazuje poštovanje koje osoba ima prema osobi kojoj piše tekst.

<span style="font-size: 17px;">6. Pritisak olovke</span>

Pritisak olovke govori o utisku koji osoba želi da ostavi, količini agresije, kreativnosti i generalne emotivnosti

<span style="font-size: 17px;">7. Nagib slova</span>

Nagib slova delimo u šest kategorija gde svaka kategorija precizno govori zasebno o svojim kategorijama.

<span style="font-size: 20px;">Kreiranje matrice parametara</span>

Kako bismo napravili adektvatnu matricu ekstraktovanih parametara pozivamo funkciju extract_features koja kao parametre prima matricu koju smo dobili korišćenjem gore navedenih metoda i niz koji ima isti broj vrsta kao matrica. Niz se sastoji od stringova u kojima piše naziv osobine koju rukopis na tom indeksu vrste poseduje u matrici. Kao rezultat dobijamo matricu dimenzija 207,7 gde svaka vrsta predstavlja jednu sliku rukopisa, a svaka kolona predstavlja jednu izvučenu karakteristiku.

In [ ]:
def extract_features(image_list):
    upper_margins = []
    pressures = []
    baseline_angles = []
    font_sizes = []
    line_spacings = []
    letter_spacings = []
    font_slants = []

    for image in image_list:
        upper_margin = extract_upper_margin(image)
        upper_margins.append(upper_margin)
        
        pressure = extract_pressure(image)
        pressures.append(pressure)
        
        baseline_angle = extract_baseline_angle(image)
        baseline_angles.append(baseline_angle)
        
        font_size = extract_font_size(image)
        font_sizes.append(font_size)

        line_spacing = extract_line_spacing(image)
        line_spacings.append(line_spacing)

        letter_spacing = extract_letter_spacing(image)
        letter_spacings.append(letter_spacing)

        font_slant = extract_font_slant(image)
        font_slants.append(font_slant)

    # Konvertovanje listi u numpy nizove
    upper_margins= np.array(upper_margins)
    pressures = np.array(pressures)
    baseline_angles = np.array(baseline_angles)
    font_sizes = np.array(font_sizes)
    line_spacings = np.array(line_spacings)
    letter_spacings = np.array(letter_spacings)
    font_slants = np.array(font_slants)

    # Kreiranje matrice sa izvučenim parametrima
    features_matrix = np.column_stack((upper_margins, pressures, baseline_angles, font_sizes, line_spacings, letter_spacings, font_slants))

    return features_matrix

<img src="https://miro.medium.com/v2/resize:fit:828/0*VsMOqHvfnPs_qa_1" alt="Opis parametara i osobina">

<span style="font-size: 20px;">SVM algoritam</span>

Support Vector Machine (SVM) je algoritam mašinskog učenja koji se koristi za klasifikaciju i regresiju. Osnovna ideja SVM-a je stvaranje optimalne hiperravne (u 2D prostoru je prava, u 3D prostoru je ravravna) koja razdvaja dve klase podataka što je bolje moguće.

Jedna od ključnih prednosti SVM-a je njegova sposobnost rada s linearno i nelinearno odvojivim podacima. Za linearno odvojive podatke, SVM koristi linearnu funkciju odlučivanja, dok za nelinearno odvojive podatke koristi transformaciju podataka u višedimenzionalni prostor (kernel trick) kako bi pronašao linearno odvojivu hiperravnu.

SVM algoritam je vrlo popularan zbog svoje sposobnosti razdvajanja podataka i dobre generalizacije na nepoznate primere. Međutim, SVM može biti osetljiv na velike skupove podataka i zahteva pažljivo podešavanje parametara kako bi postigao najbolje rezultate.

<span style="font-size: 20px;">Skaliranje podataka za SVM</span>

Kako su dobijene vrednosti u matrici prosečne vrednosti veličine slova, prosečni ugao nagiba, ... potrebno je da skaliramo podatke kako bismo primenili algoritam učenja. Ovim eliminišemo razlike u opsezima, uklanjamo uticaj ekstremnih vrednosti, poboljšavamo konvergenciju algoritma, ...

Skaliranje koje sam odabrala za SVM algoritam je:

Normalization of Features.

Baseline:<br>	0 = descending<br>
            1 = ascending<br>
            2 = straight<br><br>
Top Margin:<br>	0 = medium or bigger<br>
            1 = narrow<br><br>
Letter Size:<br>	0 = big<br>
            1 = small<br>
            2 = medium<br><br>
Line Spacing:<br>0 = big<br>
            1 = small<br>
            2 = medium<br><br>
Word Spacing:<br>	0 = big<br>
            1 = small<br>
            2 = medium<br><br>
Pen Pressure:<br>	0 = heavy<br>
            1 = light<br>
            2 = medium<br><br>
Slant Angle:<br>	0 = very reclined<br>
            1 = a little of moderately reclined <br>
            2 = a little inclined<br>
            3 = moderately inclined <br>
            4 = extremely inclined <br>
            5 = straight<br>
            6 = irregular<br><br>

<span style="font-size: 16px;">Implementacija SVM algoritma</span>

In [ ]:
def train_svm_model(data, targets):
    # Podela podataka na training_set i test_set
    X_train, X_test, y_train, y_test = train_test_split(data, targets, test_size=0.33, random_state=42)

    # Kreiranje SVM modela sa RBF kernelom
    model = SVC(kernel='rbf')

    # Treniraj model
    model.fit(X_train, y_train)

    # Prikaži tačnost algoritma
    accuracy = model.score(X_test, y_test)

    return model, accuracy

Tačnost algoritma nad datim data setom je 43%

<span style="font-size: 20px;">KNN algoritam</span>

K-Nearest Neighbors (KNN) je jednostavan i intuitivan algoritam mašinskog učenja koji se koristi za klasifikaciju i regresiju. Temelji se na ideji da slični primeri imaju tendenciju da pripadaju istoj klasi.

KNN algoritam klasifikacije radi na principu traženja K najbližih suseda (primera) za dati testni primjer. "K" predstavlja broj suseda koje algoritam uzima u obzir. Nakon što pronađe K najbližih suseda, KNN algoritam određuje klasu ili vrednost ciljnog atributa na temelju većinskog glasanja (u slučaju klasifikacije) ili prosječne vrednosti (u slučaju regresije) susednih primjera.

KNN se može koristiti za klasifikaciju i regresiju, a može se primeniti na probleme s različitim vrstama podataka. Algoritam je posebno dobar u situacijama kada imamo mnogo primera za trening i kada su slični primeri često grupisani zajedno.

Međutim, KNN algoritam ima nekoliko izazova. Efikasnost KNN-a opada s rastućim brojem primera u skupu podataka jer zahteva upoređivanje svih primera tokom predviđanja. Osim toga, odabir odgovarajućeg broja suseda (K) može biti izazovan i može imati značajan uticaj na performanse algoritma.

<span style="font-size: 20px;">Standardizacija podataka za SVM</span>

Kod KNN algoritma kao alat za skaliranje podataka najbolje je koristiti klasičnu standardizaciju. Standardizacija transformiše podatke tako da imaju srednju vrednost 0 i standardnu devijaciju 1. Ova tehnika je dobra kada želite da se rešite bilo kakvih linearnih veza između atributa i želite da dobijete podatke koji su distribuirani oko srednje vrednosti. Koristimo bilbioteku sklearn.preprocessing.py

In [ ]:
standardizator = StandardScaler()
standardizovani_podaci = standardizator.fit_transform(podaci)

<span style="font-size: 16px;">Implementacija KNN algoritma</span>

In [ ]:
def treniraj_knn_model(podaci, ciljevi):


    # Podela podataka na trening i test skup
    X_train, X_test, y_train, y_test = train_test_split(podaci, ciljevi, test_size=0.33, random_state=42)

    # Kreiranje KNN modela sa brojem suseda (K) postavljenim na 3
    model = KNeighborsClassifier(n_neighbors=3)

    # Treniranje modela
    model.fit(X_train, y_train)

    # Procena tačnosti modela na test skupu
    tacnost = model.score(X_test, y_test)

    return model, tacnost

Tačnost algoritma nad datim data setom je 37%

<span style="font-size: 20px;">Evaluacija</span>

Prilikom istraživanja najverovatnije je došlo do greške iz dva razloga:

1. DataSet je jako mali i sastoji se samo od 207 slika. Kod KNN algoritma poželjno je raditi sa što većim datasetom.
2. Metode za ektraktovanje parametara su nepouzdane i potrebno ih je nadograditi.

Sledeći korak pri istraživanju?

1. Pronaći veći data set.
2. Unaprediti metode za ekstraktovanje parametara

<span style="font-size: 20px;">Predikcije rukopisa studenata smera Informacioni inženjering</span>

Koristeći modele dobijene primenom navedenih algoritama predviđamo tipove ličnosti na smeru Informacioni inženjering.

In [ ]:

# Primena modela na preprocesirane slike
in_ekstraktovano = extract.extract_features(in_images)
in_transformisano = extract.transform_columns(in_ekstraktovano)
rezultati = model_svm.predict(in_transformisano)

print("\nPredikcije smera prema SVM algoritmu:/n")
# Prikazivanje rezultata
for i, klasa in enumerate(rezultati):
    print("Slika {} pripada klasi {}".format(i, klasa))
    
    
# Primena modela na preprocesirane slike
in_ekstraktovano = extract.extract_features(in_images)
in_transformisano = extract.transform_columns(in_ekstraktovano)
rezultati = model_knn.predict(in_transformisano)
print("\nPredikcije smera prema KNN algoritmu:/n")

# Prikazivanje rezultata
for i, klasa in enumerate(rezultati):
    print("Slika {} pripada klasi {}".format(i, klasa))

Dobijeni rezultati prema SVM-u su:

Aleksa Simeunović - Otvorenost

Ana Parović - Otvorenost

Andrej Anišić - Otvorenost

Dušan Stojanović - Prijatnost

Isidora Stančulović - Otvorenost

Nemanja Ranitović - Otvorenost

Nemanja Todorović - Otvorenost

Nikola Radović - Otvorenost

Vladimir Blanuša - Prijatnost


Dobijeni rezultati prema KNN-u su:

Aleksa Simeunović - Prijatnost

Ana Parović - Otvorenost

Andrej Anišić - Otvorenost

Dušan Stojanović - Prijatnost

Isidora Stančulović - Otvorenost

Nemanja Ranitović - Prijatnost

Nemanja Todorović - Otvorenost

Nikola Radović - Otvorenost

Vladimir Blanuša - Prijatnost

<br><span style="font-size: 20px;">Reference</span>

[1] D. J. Antony. Personality Profile Through Handwriting Analysis. Anugraha Publications, 2008.